# M4 — VLA 调用 Robo-UniLabOS 信息层闭环 demo

演示一个外部策略(VLA/ACT/经典)如何把 UniLabOS 当作**信息层**调用,完成
`affordance → pose → action_schema → (执行) → verification` 的闭环。

本 notebook 用 `local_transport`(进程内)跑通,**无需起 ROS2**。要接真实 edge,
只需把 transport 换成 `ros2_transport()`(见最后一节),并让 edge 发布 `/joint_states`、
`/resource_pose` 等,`RosLiveSource` 会把实时态喂进同一套 query。

## 1. 组装 query 引擎(实时源 + 静态源)+ 服务 + 远程客户端

In [ ]:
from typing import List, Optional

from unilabos.api import QueryService
from unilabos.queries.models import QueryAffordance
from unilabos.queries.ros_live_source import build_live_query_engine
from unilabos_client import RoboUniLabOSRemote, local_transport


# 一个极简静态场景源:提供 balance_1 的 tare 按钮 affordance(真实部署来自 LSD/LabUtopia)
class DemoSceneSource:
    name = "demo_scene"

    def query_pose(self, target, frame=None):
        return None

    def query_state(self, target):
        return None

    def query_affordance(self, target, kind=None) -> List[QueryAffordance]:
        if target == "balance_1":
            return [QueryAffordance(id="tare_btn", kind="button",
                                    action_primitives=["press_button"],
                                    target="balance_1.tare_button")]
        return []

    def query_action_schema(self, action):
        return None

    def query_safety_zones(self):
        return []


live, engine = build_live_query_engine(static_sources=[DemoSceneSource()])
service = QueryService(engine)
client = RoboUniLabOSRemote(local_transport(service))
print("query engine + service + remote client ready")

## 2. 模拟 edge 实时态

真实运行时,这些由 edge 的 `/joint_states`、`/resource_pose`、TwinBridge 自动喂入
`RosLiveSource`。这里手动喂入以离线演示。

In [ ]:
live.update_pose("balance_1.tare_button", [0.60, 0.0, 0.07], frame_id="robot_base")
live.update_state("balance_1", {"mass_g": 5.01, "stable": True})
print("fed live pose + state")

## 3. VLA 调用闭环:affordance → pose → action_schema → (执行) → verification

In [ ]:
import json

# (1) 这台天平上有什么可操作部位?
affs = client.query_affordance("balance_1")
print("[1] affordance:", json.dumps(affs, ensure_ascii=False))

target = affs["affordances"][0]["target"]

# (2) tare 按钮在机器人基座系下的位姿(来自 edge 实时态)
pose = client.query_pose(target)
print("[2] pose:", json.dumps(pose, ensure_ascii=False))

# (3) 按按钮这个动作的 schema(前置/后置条件、策略回退顺序)
schema = client.query_action_schema("press_button")
print("[3] action_schema:", json.dumps(schema, ensure_ascii=False))

# (4) 这里策略层执行 move_l + press(本 demo 省略,真实由 HAL/VLA 完成)
print("[4] execute: policy moves to pose and presses (stubbed)")

# (5) 后置条件验证
result = client.query_verification(task_id="press_tare_demo", action="press_button")
print("[5] verification:", json.dumps(result, ensure_ascii=False))

## 4. 接真实 edge:把 transport 换成 ROS2

edge 侧(`unilab --mode sim/twin/real`)需启动 `QueryServiceNode`(暴露 `/unilabos/query`),
并用 `build_live_query_engine(node=...)` 把 `RosLiveSource` 订阅到 `/joint_states` 等。
客户端只改一行:

```python
from unilabos_client import RoboUniLabOSRemote, ros2_transport
client = RoboUniLabOSRemote(ros2_transport(service_name="/unilabos/query"))
# 之后 client.query_pose(...) 等用法完全一致
```

同一套 `RoboUniLabOSRemote` 代码,local ↔ ros2 ↔ (未来 gRPC) 透明切换。